import

In [2]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'   # Suppress oneDNN info messages
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'     # Suppress TF info/warning messages

import re
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

print("=" * 60)
print("TensorFlow version:", tf.__version__)
print("=" * 60)

TensorFlow version: 2.20.0


LOAD DATA

In [3]:
print("\n[1/6] Loading data...")
_df = pd.read_csv("cleaned_all_job.csv")
print(f"  Shape: {_df.shape}")
print(f"  Columns: {_df.columns.tolist()}")
print(f"  Nulls:\n{_df.isnull().sum()}")
print(f"\n  Category distribution:\n{_df['category'].value_counts()}")

_df['text_length'] = _df['job_description'].apply(len)
print(f"\n  Text length stats:\n{_df['text_length'].describe()}")


[1/6] Loading data...
  Shape: (1167, 5)
  Columns: ['job_id', 'category', 'job_title', 'job_description', 'job_skill_set']
  Nulls:
job_id             0
category           0
job_title          0
job_description    0
job_skill_set      0
dtype: int64

  Category distribution:
category
information technology    240
business development      239
finance                   236
sales                     232
hr                        220
Name: count, dtype: int64

  Text length stats:
count     1167.000000
mean      3658.147386
std       2040.926404
min        161.000000
25%       2065.500000
50%       3367.000000
75%       4798.500000
max      13244.000000
Name: text_length, dtype: float64


PREPROCESSING

In [4]:
print("\n[2/6] Preprocessing...")
_df = _df.dropna(subset=['job_title', 'job_description', 'job_skill_set', 'category'])

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

_df['text'] = (
    _df['job_title'].astype(str) + ' ' +
    _df['job_description'].astype(str) + ' ' +
    _df['job_skill_set'].astype(str)
).apply(clean_text)

print(f"  Rows after cleaning: {len(_df)}")


[2/6] Preprocessing...
  Rows after cleaning: 1167


LABEL ENCODING

In [5]:
print("\n[3/6] Label encoding...")
label_encoder = LabelEncoder()
_df['label'] = label_encoder.fit_transform(_df['category'])
num_classes = len(label_encoder.classes_)
print(f"  Classes ({num_classes}): {label_encoder.classes_.tolist()}")


[3/6] Label encoding...
  Classes (5): ['business development', 'finance', 'hr', 'information technology', 'sales']


TRAIN/TEST SPLIT

In [6]:
print("\n[4/6] Splitting data (80/20)...")
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    _df['text'],
    _df['label'],
    test_size=0.2,
    random_state=42,
    stratify=_df['label']
)

X_train = np.array(X_train_raw, dtype=object)
X_test  = np.array(X_test_raw,  dtype=object)
y_train = np.array(y_train).astype('int32')
y_test  = np.array(y_test).astype('int32')

print(f"  X_train dtype : {X_train.dtype}   ← must be 'object'")
print(f"  X_test  dtype : {X_test.dtype}")
print(f"  y_train dtype : {y_train.dtype}")
print(f"  Train size    : {len(X_train)} | Test size: {len(X_test)}")


[4/6] Splitting data (80/20)...
  X_train dtype : object   ← must be 'object'
  X_test  dtype : object
  y_train dtype : int32
  Train size    : 933 | Test size: 234


TOKENIZATION & VECTORIZATION

In [7]:
print("\n[5/6] Building TextVectorization layer...")
MAX_TOKENS  = 10_000
MAX_LENGTH  = 200
EMBED_DIM   = 128
LSTM_UNITS  = 64

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode='int',
    output_sequence_length=MAX_LENGTH
)
vectorizer.adapt(X_train)
print(f"  Vocab size (actual): {len(vectorizer.get_vocabulary())}")


[5/6] Building TextVectorization layer...
  Vocab size (actual): 10000


MODEL (Functional API)


In [8]:
print("\n[6/6] Building model...")

inputs  = tf.keras.Input(shape=(), dtype=tf.string, name="text_input")
x       = vectorizer(inputs)
x       = layers.Embedding(
              input_dim=MAX_TOKENS,
              output_dim=EMBED_DIM,
              name="embedding"
          )(x)
x       = layers.Bidirectional(layers.LSTM(LSTM_UNITS), name="bi_lstm")(x)
x       = layers.Dropout(0.3, name="dropout")(x)
x       = layers.Dense(64, activation='relu', name="dense_hidden")(x)
outputs = layers.Dense(num_classes, activation='softmax', name="output")(x)

model = Model(inputs, outputs, name="JobClassifier")

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()


[6/6] Building model...


Model: "JobClassifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_input (InputLayer)         │ (None)                 │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization              │ (None, 200)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bi_lstm (Bidirectional)         │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,387,397 (5.29 MB)

 Trainable params: 1,387,397 (5.29 MB)

 Non-trainable params: 0 (0.00 B)

TRAINING

In [9]:
print("\nStarting training...\n")

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True,
        verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=32,
    callbacks=callbacks
)


Starting training...

Epoch 1/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - accuracy: 0.7899 - loss: 1.4676 - val_accuracy: 0.8718 - val_loss: 1.0016
Epoch 2/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9625 - loss: 0.4122 - val_accuracy: 0.9915 - val_loss: 0.0732
Epoch 3/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9957 - loss: 0.0381 - val_accuracy: 0.9872 - val_loss: 0.0341
Epoch 4/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.9979 - loss: 0.0186 - val_accuracy: 1.0000 - val_loss: 0.0144
Epoch 5/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.9957 - loss: 0.0114 - val_accuracy: 0.9957 - val_loss: 0.0156
Epoch 6/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.9989 - loss: 0.0058 - val_accuracy: 1.0000 - val_loss: 0.0096
Epoch 7/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 1.0000 - loss: 0.0030 - val_accuracy: 0.9957 - val_loss: 0.0128
Epoch 7: early stopping
Restoring model weights from the end of the best epoch: 4.


EVALUATION

In [10]:
print("\n" + "=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"  Test Loss     : {loss:.4f}")
print(f"  Test Accuracy : {acc * 100:.2f}%")

best_val_acc = max(history.history['val_accuracy'])
print(f"  Best Val Acc  : {best_val_acc * 100:.2f}%")


EVALUATION RESULTS
  Test Loss     : 0.0144
  Test Accuracy : 100.00%
  Best Val Acc  : 100.00%


SAVE MODEL

In [11]:
model.save("job_classifier_model.keras")
print("\nModel saved → job_classifier_model.keras")

sample_texts = np.array([
    "data scientist python machine learning tensorflow deep learning",
    "frontend developer react javascript html css web design",
], dtype=object)

preds = model.predict(sample_texts, verbose=0)
pred_labels = label_encoder.inverse_transform(np.argmax(preds, axis=1))
print("\nSample Predictions:")
for text, label, prob in zip(sample_texts, pred_labels, preds):
    confidence = np.max(prob) * 100
    print(f"  '{text[:55]}...'  →  {label}  ({confidence:.1f}%)")

print("\nDone!")


Model saved → job_classifier_model.keras

Sample Predictions:
  'data scientist python machine learning tensorflow deep ...'  →  business development  (94.2%)
  'frontend developer react javascript html css web design...'  →  business development  (83.4%)

Done!
